In [40]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "amici2008fission")
original_data_pathway = os.path.join(pathway, "original_data")
complete_path_1 = os.path.join(original_data_pathway, "Amici_2008_CurrBiol_FIES_bias_update.csv")

In [41]:
import pandas as pd
import numpy as np
df = pd.read_csv(complete_path_1)


In [42]:
df['study_id']="amici2008fission"
df['experiment_name']="delay_of_gratification_task"
df['trial'] = '10'


In [43]:
df.columns = map(str.lower, df.columns)
df = df.applymap(lambda s: s.lower() if type(s) == str else s)
df = df.rename(columns={"subject": "ape",
    "delay (ss)": "delay_in_seconds",
    "larger amount chosen (number, out of the 10 free-choice trials)": "larger_amount_chosen_out_of_10_free_choice_trials",
    "bias (out of the 10 free-choice trials)": "bias_out_of_10_free_choice_trials"})


# df.columns

In [44]:
df['date']= pd.to_datetime(df['date'],format='%m/%d/%Y')

df['year']= df['date'].dt.year
df['month']= df['date'].dt.month
df['day']= df['date'].dt.day
# df.columns

In [45]:
comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
new_df= df.merge(apedf,left_on='ape', right_on='name', how='left')

new_df = new_df.rename(columns={"species_y": "species"})

# new_df.columns
new_df.rename(columns={"ape": "participant",
                "indifference point":"indifference_point"}, inplace=True)

import re
replace_1=re.compile('\([^)]*\)')
new_df['indifference_point'].replace(replace_1, '', inplace=True, regex=True)
# new_df['indifference_point'].replace('\)', '', inplace=True, regex=True)
new_df['indifference_point']=new_df['indifference_point'].str.rstrip()
new_df['indifference_point'].replace(' ', '_', inplace=True, regex=True)
new_df['consequences'].replace(' = ', '=', inplace=True, regex=True)
new_df['consequences']=new_df['consequences'].str.rstrip()
new_df['consequences'].replace(' ', '_', inplace=True, regex=True)

In [46]:
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 

new_df= new_df.merge(ape_dob,left_on='participant', right_on='name', how='left') #insert dob of participants
new_df['dodc'] = new_df['year'].astype(str) + '-' + new_df['month'].astype(str) + '-' + new_df['day'].astype(str)
new_df['dodc'] = pd.to_datetime(new_df['dodc'])
new_df['dob'] = pd.to_datetime(new_df['dob'])

new_df['age_in_years'] = (new_df['dodc'] - new_df['dob']).dt.days//365 

In [47]:
amici2008fission_standardized=new_df[['study_id','experiment_name', 'year', 'month', 'day',  
        'participant', 'age_in_years','sex', 'species',  'session', 
        'delay_in_seconds',
       'larger_amount_chosen_out_of_10_free_choice_trials',
       'consequences', 
       'indifference_point']]

In [48]:
out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)
comp_out_path_stand = os.path.join(out_pathway, 'amici2008fission_standardized.csv')
amici2008fission_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

In [49]:
# ape_study_working = df[['study_id',  'ape']]

# file = open("ape_study_working.csv", "w")
# file.write(ape_study_working.to_csv()) 
# file.close()

In [50]:
names =amici2008fission_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
amici2008fission_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'amici2008fission_glossary.csv')
amici2008fission_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
